# Benchmark MiniZinc - Notebook

Ce notebook permet d'executer le benchmark et de visualiser directement les resultats (CSV + courbes).

In [20]:
# Installation des dependances dans le kernel courant
from pathlib import Path
import subprocess
import sys

req = Path('python/benchmarking/requirements.txt')
if not req.exists():
    req = Path('requirements.txt')

print(f'Requirements utilises: {req.resolve()}')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(req)])


Requirements utilises: /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/requirements.txt


0

In [21]:
from pathlib import Path
from types import SimpleNamespace
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'package.json').exists():
            return candidate
    raise RuntimeError('Impossible de localiser la racine du monorepo depuis le dossier courant.')

PROJECT_ROOT = find_repo_root(Path.cwd().resolve())
BENCH_DIR = PROJECT_ROOT / 'python' / 'benchmarking'
print('PROJECT_ROOT =', PROJECT_ROOT)
print('BENCH_DIR    =', BENCH_DIR)

sys.path.append(str(BENCH_DIR))
from benchmark_pipeline import run_benchmark


PROJECT_ROOT = /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1
BENCH_DIR    = /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking


## Generation d'instances parametrees (A/R/K/T)\n
Active cette section pour creer une campagne de tailles avant le benchmark.

In [26]:
# Optionnel: generer une campagne de tailles\n
import subprocess, sys
GENERATE_SCALING_INSTANCES = True
if GENERATE_SCALING_INSTANCES:
    cmd = [
        sys.executable,
        str(BENCH_DIR / 'generate_soutenance_instances.py'),
        '--output-dir', str(BENCH_DIR / 'test_files' / 'scaling'),
        '--A-list', '5,8,10,12,15',
        '--R-list', '15,24,30,36,45',
        '--K-list', '3,4,4,5,6',
        '--T-list', '5,6,6,8,8',
        '--mode', 'zipped',
    ]
    print(' '.join(cmd))
    subprocess.check_call(cmd)
else:
    print('Generation ignoree (GENERATE_SCALING_INSTANCES=False)')

/home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/.venv/bin/python /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/generate_soutenance_instances.py --output-dir /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/test_files/scaling --A-list 5,8,10,12,15 --R-list 15,24,30,36,45 --K-list 3,4,4,5,6 --T-list 5,6,6,8,8 --mode zipped
/home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/test_files/scaling/soutenance_A5_R15_K3_T5.planning
/home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/test_files/scaling/soutenance_A8_R24_K4_T6.planning
/home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/test_files/scaling/soutenance_A10_R30_K4_T6.planning
/home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/test_files/scaling/soutenance_A12_R36_K5_T8.pla

In [27]:
# Parametres de campagne
INPUTS = [str(BENCH_DIR / 'test_files/scaling')]
INPUT_DIR_LEGACY = []
OUTPUT_CSV = BENCH_DIR / 'result' / 'benchmark_results.csv'
RESULT_DIR = BENCH_DIR / 'result'
GENERATED_MZN_DIR = BENCH_DIR / 'generated_mzn'
TIMEOUT_SECONDS = 300  # 5 minutes
REPO_ROOT = PROJECT_ROOT
SKIP_BACKEND_BUILD = False
MINIZINC_BIN = Path('/snap/minizinc/current/bin/minizinc')
INCLUDE_OPTAPLANNER = False
KEEP_GENERATED = True
KEEP_RESULT = False

print('INPUTS =', INPUTS)
print('RESULT_DIR =', RESULT_DIR)


INPUTS = ['/home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/test_files/scaling']
RESULT_DIR = /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/python/benchmarking/result


In [28]:
args = SimpleNamespace(
    inputs=INPUTS,
    input_dir=INPUT_DIR_LEGACY,
    output_csv=OUTPUT_CSV,
    result_dir=RESULT_DIR,
    generated_mzn_dir=GENERATED_MZN_DIR,
    timeout=TIMEOUT_SECONDS,
    repo_root=REPO_ROOT,
    skip_backend_build=SKIP_BACKEND_BUILD,
    minizinc_bin=MINIZINC_BIN,
    include_optaplanner=INCLUDE_OPTAPLANNER,
    keep_generated=KEEP_GENERATED,
    keep_result=KEEP_RESULT,
)

await run_benchmark(args)


[Env] MiniZinc force via --minizinc-bin: /snap/minizinc/1257/bin/minizinc
[Env] Version active: MiniZinc to FlatZinc converter, version 2.9.7, build 2490193457
[Env] minizinc-python driver: /snap/minizinc/1257/bin/minizinc

[Backend] Build du CLI planning-spec-cli...
.                                        |  WARN  Unsupported engine: wanted: {"node":">=20.10.0"} (current: {"node":"v18.19.1","pnpm":"10.31.0"})
backend/packages/cli                     |  WARN  Unsupported engine: wanted: {"node":">=20.10.0"} (current: {"node":"v18.19.1","pnpm":"10.31.0"})
backend/packages/language                |  WARN  Unsupported engine: wanted: {"node":">=20.10.0"} (current: {"node":"v18.19.1","pnpm":"10.31.0"})
backend/packages/server                  |  WARN  Unsupported engine: wanted: {"node":">=20.10.0"} (current: {"node":"v18.19.1","pnpm":"10.31.0"})

> planning-spec-cli@0.0.1 build /home/lomofouet/Documents/Recherches M2/IMPLEMENTATION_MONO_REPO_V1/backend/packages/cli
> tsc -p tsconfig.json

CancelledError: 

In [ ]:
df = pd.read_csv(RESULT_DIR / 'benchmark_results.csv')
summary_df = pd.read_csv(RESULT_DIR / 'solver_summary.csv')
status_df = pd.read_csv(RESULT_DIR / 'status_by_solver.csv')

print(f'Nombre de lignes benchmark: {len(df)}')
if df.empty:
    print('Aucun resultat: verifier la cellule de run et les messages [Erreur]/[Warn].')
else:
    display(df.head(20))

display(summary_df)
display(status_df)


In [ ]:
# Afficher les courbes generees automatiquement par le pipeline
pngs = sorted(RESULT_DIR.glob('curve_*.png'))
print(f"Nombre d'images trouvees: {len(pngs)}")
if not pngs:
    print('Aucune image. Soit le benchmark est vide, soit matplotlib etait indisponible.')
else:
    for png in pngs:
        print(png.name)
        display(Image(filename=str(png.resolve())))


In [ ]:
# Exemple de visualisation complementaire
if df.empty:
    print('Visualisation ignoree: dataframe vide.')
else:
    pivot = df.pivot_table(index='Instance', columns='Solveur', values='T_total', aggfunc='mean')
    if pivot.empty:
        print('Pivot vide: pas de T_total exploitable.')
    else:
        ax = pivot.plot(kind='bar', figsize=(14, 6))
        ax.set_title('Temps total moyen par instance et solveur')
        ax.set_ylabel('Secondes')
        plt.tight_layout()
        plt.show()
